# Notebook 3 – Parameter Optimization
> Find the best hyperparameters using grid, random, or Bayesian search.
>
> **Educational only. Past performance does not guarantee future results.**

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import matplotlib.pyplot as plt
from data.fetcher import fetch_yfinance
from src.strategies import MeanReversionStrategy
from src.optimizer import StrategyOptimizer
from src.utils import plot_optimization_heatmap

In [ ]:
df = fetch_yfinance('AAPL', start='2015-01-01', end='2023-01-01')

param_space = {
    'lookback':       [10, 15, 20, 30, 50],
    'rsi_period':     [10, 14, 21],
    'rsi_oversold':   [25, 30, 35],
    'rsi_overbought': [65, 70, 75],
}

opt = StrategyOptimizer(
    strategy_cls=MeanReversionStrategy,
    df=df,
    param_space=param_space,
    metric='sharpe',
    method='grid',
    cv_folds=5,
)
best_params, results_df = opt.run()
print('Best params:', best_params)
results_df.head(10)

In [ ]:
fig = plot_optimization_heatmap(
    results_df,
    param_x='lookback',
    param_y='rsi_period',
    metric='sharpe',
    title='Sharpe Heatmap – Mean Reversion · AAPL',
)
plt.show()

In [ ]:
# Bayesian search – far more efficient for large spaces
opt_bayes = StrategyOptimizer(
    strategy_cls=MeanReversionStrategy,
    df=df,
    param_space=param_space,
    metric='sharpe',
    method='bayesian',
    n_trials=100,
    cv_folds=5,
)
best_bayes, res_bayes = opt_bayes.run()
print('Bayesian best params:', best_bayes)

In [ ]:
# Convergence plot
res_bayes = res_bayes.reset_index(drop=True)
res_bayes['best_so_far'] = res_bayes['sharpe'].cummax()
plt.figure(figsize=(10, 4))
plt.plot(res_bayes.index, res_bayes['best_so_far'], color='steelblue', lw=2)
plt.xlabel('Trial'); plt.ylabel('Best Sharpe Ratio')
plt.title('Bayesian Optimization Convergence')
plt.grid(alpha=0.3)
plt.show()